# Laya medical full fine-tune (T4×2)\n\nClones the public repo and runs `scripts/run_full.sh` (~30k Q × 3 epochs, per-epoch dual eval, wandb).\n\n**Settings:** Accelerator = GPU T4 x2, Internet = On. Attach Kaggle secret `wandb_api_key` (or `WANDB_API_KEY`).

In [ ]:
import os, sys, platform, subprocess, shutil, json
from pathlib import Path
print('python', sys.version)
print('platform', platform.platform())
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
n = torch.cuda.device_count()
print('n_gpu', n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}', p.name, f'{p.total_memory/1e9:.1f}GB')
if n < 1:
    raise SystemExit('No GPU visible. Set Accelerator to GPU T4 x2.')
os.environ['USE_TF'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# Load wandb API key from Kaggle User Secrets (never print the value).
secret_names_tried = []
loaded_from = None
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for name in ('wandb_api_key', 'WANDB_API_KEY', 'wandb'):
        secret_names_tried.append(name)
        try:
            val = client.get_secret(name)
        except Exception:
            continue
        if val:
            os.environ['WANDB_API_KEY'] = val
            loaded_from = name
            break
except Exception as e:
    print('kaggle_secrets error:', type(e).__name__, e)
print('wandb secret tried:', secret_names_tried)
print('wandb secret loaded_from:', loaded_from)
print('WANDB_API_KEY present:', bool(os.environ.get('WANDB_API_KEY')))
if not os.environ.get('WANDB_API_KEY'):
    raise SystemExit('Missing wandb secret. Add User Secret wandb_api_key (or WANDB_API_KEY) and attach it to this notebook.')

In [ ]:
%pip -q install -U 'laya>=0.3.7' 'transformers>=4.48.0' 'datasets>=3.0.0' safetensors huggingface_hub accelerate scipy pyarrow pandas tabulate PyYAML wandb

In [ ]:
REPO = 'https://github.com/Priyanshu-5257/laya-medical-finetune.git'
BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/repo')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO, str(REPO_DIR)])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'])

In [ ]:
def run(cmd, cwd=None, env=None):
    print('+', *cmd, flush=True)
    e = os.environ.copy()
    if env:
        e.update(env)
    p = subprocess.Popen(cmd, cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(rc)

run(['bash', 'scripts/run_full.sh'], cwd=str(REPO_DIR), env={
    'WORK': '/kaggle/working',
    'CONFIG': 'configs/full.yaml',
})

In [ ]:
summary = Path('/kaggle/working/summary_full.json')
report = Path('/kaggle/working/eval_report_full.json')
print(summary.read_text() if summary.exists() else 'MISSING summary_full.json')
print('---')
print(report.read_text()[:6000] if report.exists() else 'MISSING eval_report_full.json')
hist = Path('/kaggle/working/laya_medical_full/train_history.json')
print('--- history ---')
print(hist.read_text() if hist.exists() else 'MISSING train_history.json')